# Guerrero's nonzero difference mean

The general published PLS penalty is ||y−τ||² + λ||D_d τ−μ1||². Holding μ fixed is distinct from inserting Guerrero's observed-difference plug-in μ̂=mean(D_d y).

In [ ]:
import awesomefigures as af
from matplotlib_inline.backend_inline import set_matplotlib_formats

# Publication styling only; numerical estimates and metrics are unchanged.
af.set_style("elegante", font="DejaVu Sans", use_tex=False,
             palette="okabe-ito", rasterize=False)
set_matplotlib_formats("svg")


## Illustrate a trending signal

When d=1 and μ≈slope, a strong penalty preserves a fitted line with the specified increment; μ=0 instead penalizes all slope.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from trendestimation import TrendSmoother
rng=np.random.default_rng(15);t=np.arange(90)
truth=2+.12*t;y=truth+rng.normal(0,.35,len(t))
sm=TrendSmoother(len(y),order=1)
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(y,color="0.8",label="observed");ax.plot(truth,"k:",label="true")
for mu in [0,.12,"data"]:
    f=sm.fit(y,lambda_=1000,mu=mu)
    ax.plot(f.trend,label=f"μ={mu}, used={f.mu:.3f}")
ax.set(title="Mean first difference changes limiting behavior");ax.legend();plt.show()

## Affine null-space limit

At λ=∞, D_d τ=μ1. For d=2, a nonzero μ generates quadratic curvature instead of the linear null-space trend associated with μ=0.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,3.5))
for a,d in zip(axs,[1,2]):
    model=TrendSmoother(len(y),d)
    for mu in [0,.02]:
        fitted=model.fit(y,lambda_=float("inf"),mu=mu)
        a.plot(fitted.trend,label=f"μ={mu}")
        assert np.max(np.abs(model.D@fitted.trend-mu))<1e-6
    a.set(title=f"order d={d}, exact λ=∞",xlabel="Observation");a.legend()
fig.tight_layout();plt.show()

## Conditional EDF and limits

The displayed EDF=trace(Hλ) conditions on μ. It is not the full EDF of the estimator if μ was selected from the same sample; classical selectors in this release require fixed numeric μ.

In [ ]:
s=np.geomspace(.001,1e6,60)
fig,ax=plt.subplots(figsize=(8,3.5))
for d in [1,2,3]:
    model=TrendSmoother(len(y),d)
    ax.semilogx(s,[model.degrees_of_freedom(float(v)) for v in s],label=f"d={d}")
ax.set(xlabel="λ",ylabel="Conditional EDF",title="Effective degrees of freedom")
ax.legend();plt.show()
print("μ='data' is allowed for fitting, not for fixed-μ CV/GCV/AICc/BIC selection.")

## Difference-mean plug-in with Gaussian MLE

The mode `mu="data"` uses the **sample average of the observed (d)-th differences**, `np.mean(D_d @ y)`. In Gaussian MLE, the likelihood conditions on this plug-in; it is **not** a joint maximum-likelihood estimate of `mu`, `lambda`, and the variances. The two variance components measure independent observation noise and the latent centered differences—not the variance of the original series or of its fitted trend.

In [ ]:
from trendestimation import select_lambda, estimate_variance_components

solver = TrendSmoother(len(y), order=1)
difference_mean = float(np.mean(solver.D @ y))
selected = select_lambda(solver, y, criterion="mle", mu="data")
components = estimate_variance_components(
    solver, y, lambda_=selected.lambda_, mu="data"
)
np.testing.assert_allclose(components.mu, difference_mean)
print(f"Empirical first-difference mean: {components.mu:.6f}")
print(f"Selected MLE lambda: {selected.lambda_:.5g}")
print(f"Observation-noise variance: {components.observation_variance:.5g}")
print(f"Latent first-difference variance: {components.difference_variance:.5g}")
print("These are conditional Gaussian model estimates, not residual MSE.")
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(y, ".", alpha=0.35, label="Observed")
ax.plot(truth, "k--", label="Synthetic truth")
ax.plot(solver.fit(y, lambda_=selected.lambda_, mu="data").trend,
        label="MLE, observed-difference mu")
ax.set(xlabel="Time", ylabel="Value", title="Guerrero plug-in mean and Gaussian MLE")
ax.legend()
plt.show()